# 🤖 Notebook 02 — Model Comparison

**Khóa luận:** Phân tích Lan tỏa Rủi ro sử dụng NOTEARS Causal Discovery  
**Chương tương ứng:** Chương 4.2 → 4.5

---

## Mục tiêu
1. Chạy 3 phương pháp: Granger (baseline) → PC Algorithm → NOTEARS
2. So sánh theo SHD, Precision, Recall, F1 + Bootstrap CI 95%
3. Phân tích threshold sensitivity cho NOTEARS
4. Vẽ DAG cho từng phương pháp
5. Sinh bảng LaTeX cho khóa luận

## 0. Setup

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.logger import setup_logger, LogTimer
setup_logger('KLTN')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
print('✅ Setup hoàn tất')

---
## 1. Load dữ liệu (từ Notebook 01)

In [ ]:
from src.data.collector    import DataCollector
from src.data.preprocessor import DataPreprocessor
from config import NODE_NAMES, MARKET_LABELS

collector = DataCollector(vnindex_path='../data/raw/vnindex_manual.csv')
prices    = collector.fetch()

prep   = DataPreprocessor()
X, vol = prep.fit_transform(prices, standardize=True)
vol_df = prep.volatility_

print(f'X shape: {X.shape} | Nodes: {NODE_NAMES}')

---
## 2. Chương 4.2 — Granger Causality (Baseline)

In [ ]:
from src.models.granger import GrangerCausality

with LogTimer('Granger Causality'):
    granger = GrangerCausality(max_lag=5, node_names=NODE_NAMES)
    granger_result = granger.fit_timed(X)

print(granger.summary())
print('\n📋 P-value matrix:')
granger.pvalue_df().style.background_gradient(cmap='RdYlGn_r', vmin=0, vmax=0.1).format('{:.4f}')

In [ ]:
print('\n📌 Cặp có quan hệ Granger (p < 0.05):')
pairs = granger.significant_pairs()
for p in pairs:
    cause  = MARKET_LABELS.get(p['cause'],  p['cause'])
    effect = MARKET_LABELS.get(p['effect'], p['effect'])
    print(f'  {cause:<15s} → {effect:<15s} | p={p["p_val"]:.4f} | F={p["F_stat"]:.3f}')

In [ ]:
from src.visualization.causal_graph import plot_dag

fig = plot_dag(
    binary_matrix = granger_result.binary_matrix,
    node_names    = NODE_NAMES,
    title         = 'Granger Causality — Causal DAG (Baseline)',
    output_path   = '../reports/figures/02_dag_granger.png',
    show_weights  = False,
)
plt.show()

---
## 3. Chương 4.3 — PC Algorithm

In [ ]:
from src.models.pc_algorithm import PCAlgorithm

with LogTimer('PC Algorithm'):
    pc     = PCAlgorithm(alpha=0.05, node_names=NODE_NAMES)
    pc_result = pc.fit_timed(X)

print(pc.summary())
orient = pc.orientation_summary()
print(f'\n📌 Orientation: directed={orient["directed"]} | '
      f'undirected={orient["undirected"]} | rate={orient["orientation_rate"]:.1%}')

In [ ]:
fig = plot_dag(
    binary_matrix = pc_result.binary_matrix,
    node_names    = NODE_NAMES,
    title         = 'PC Algorithm — Causal DAG (CPDAG)',
    output_path   = '../reports/figures/02_dag_pc.png',
    show_weights  = False,
)
plt.show()

---
## 4. Chương 4.4 — NOTEARS (Thuật toán chính)

In [ ]:
# 4.1 Threshold Sensitivity trước khi chọn threshold
from src.models.notears import NOTEARS
from src.visualization.time_series import plot_threshold_sensitivity

# Chạy với threshold=0 để lấy W_raw
notears_raw = NOTEARS(lambda1=0.1, threshold=0.0, node_names=NODE_NAMES)
notears_raw.fit_timed(X)

sensitivity = notears_raw.threshold_sensitivity()
fig = plot_threshold_sensitivity(
    sensitivity = sensitivity,
    highlight   = 0.2,
    output_path = '../reports/figures/02_threshold_sensitivity.png',
)
plt.show()

print('\n📋 Bảng threshold sensitivity:')
pd.DataFrame(sensitivity)

In [ ]:
# 4.2 Chạy NOTEARS với threshold tối ưu
BEST_THRESHOLD = 0.2   # chọn từ sensitivity analysis ở trên

with LogTimer('NOTEARS'):
    notears = NOTEARS(lambda1=0.1, threshold=BEST_THRESHOLD, node_names=NODE_NAMES)
    notears_result = notears.fit_timed(X)

print(notears.summary())
print('\n📋 Degree table:')
notears_result.degree_table()

In [ ]:
# 4.3 Adjacency matrix heatmap
from src.visualization.causal_graph import plot_dag, plot_adjacency_heatmap

fig = plot_adjacency_heatmap(
    weight_matrix = notears_result.adjacency_matrix,
    node_names    = NODE_NAMES,
    title         = f'NOTEARS — Ma trận trọng số W (threshold={BEST_THRESHOLD})',
    output_path   = '../reports/figures/02_notears_adjacency_heatmap.png',
)
plt.show()

In [ ]:
# 4.4 DAG sau threshold
fig = plot_dag(
    binary_matrix = notears_result.binary_matrix,
    node_names    = NODE_NAMES,
    weight_matrix = notears_result.adjacency_matrix,
    title         = f'NOTEARS ★ — Causal DAG (threshold={BEST_THRESHOLD})',
    output_path   = '../reports/figures/02_dag_notears.png',
)
plt.show()

In [ ]:
# 4.5 Degree analysis
from src.visualization.risk_map import plot_degree_analysis

fig = plot_degree_analysis(
    result      = notears_result,
    node_names  = NODE_NAMES,
    title       = 'NOTEARS ★ — In-degree vs Out-degree',
    output_path = '../reports/figures/02_notears_degree.png',
)
plt.show()

---
## 5. Chương 4.5 — So sánh 3 phương pháp

In [ ]:
from src.evaluation.comparator import MethodComparator

# Ground truth = Granger (pseudo-ground-truth)
comparator = MethodComparator(
    true_dag    = granger_result.binary_matrix,
    n_bootstrap = 1000,
)
comparator.add('PC Algorithm', pc_result)
comparator.add('NOTEARS ★',   notears_result)

comparator.print_summary()

In [ ]:
# Bảng so sánh đầy đủ
df_compare = comparator.comparison_table()
df_compare

In [ ]:
# Bar chart so sánh
fig = comparator.plot_comparison(
    output_path='../reports/figures/02_comparison_bar_chart.png'
)
plt.show()

In [ ]:
# LaTeX table
latex = comparator.latex_table()
print('📄 LaTeX code (dán vào khóa luận):\n')
print(latex)

# Lưu file .tex
with open('../reports/tables/02_method_comparison.tex', 'w', encoding='utf-8') as f:
    f.write(latex)
print('\n✅ Đã lưu: reports/tables/02_method_comparison.tex')

---
## 6. Lưu kết quả

In [ ]:
from src.utils.io_utils import ResultIO

io = ResultIO('../reports')

results = {
    'Granger'     : granger_result,
    'PC Algorithm': pc_result,
    'NOTEARS'     : notears_result,
}

io.save_all_results(results)
io.save_comparison(df_compare, filename='02_method_comparison')
io.snapshot(results, df_compare, vol_df)

print('\n✅ Đã lưu snapshot — Notebook 03 có thể load trực tiếp')